# 6. Hyperparameter Tuning — CKD Prediction

**Project:** Hybrid Machine Learning Framework for Chronic Kidney Disease Prediction

### Objective
Tune 9 classification algorithms using **RandomizedSearchCV with 3 sampled configurations per algorithm** and **5-fold Stratified Cross-Validation**.

This version is intentionally lightweight for the 400-record CKD dataset while preserving leakage-safe preprocessing and an untouched test set.


In [3]:
# 1. Imports
import os
import warnings
import importlib.util
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_ITER = 3
CV_FOLDS = 5

print("Imports successful.")
print(f"RandomizedSearchCV iterations per algorithm: {N_ITER}")
print(f"Cross-validation folds: {CV_FOLDS}")


Imports successful.
RandomizedSearchCV iterations per algorithm: 3
Cross-validation folds: 5


In [4]:
# 2. Load the ORIGINAL CKD CSV
candidate_paths = [
    "../../dataset/raw/chronic_kidney_disease.csv",
    "../dataset/raw/chronic_kidney_disease.csv",
    "dataset/raw/chronic_kidney_disease.csv",
    "chronic_kidney_disease.csv"
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_path is None:
    raise FileNotFoundError(
        "Original chronic_kidney_disease.csv was not found. "
        "Place it in dataset/raw/ and rerun this cell."
    )

df = pd.read_csv(data_path)

print("Dataset:", data_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

assert df.shape == (400, 25), f"Expected (400,25), found {df.shape}"
assert "class" in df.columns

print("\nDataset verification PASSED: 400 rows × 25 columns.")


Dataset: ../../dataset/raw/chronic_kidney_disease.csv
Shape: (400, 25)
Columns: ['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']

Dataset verification PASSED: 400 rows × 25 columns.


In [5]:
# 3. Target and predictor setup
df = df.copy()

# Clean string whitespace while preserving missing values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Map target: CKD = 1, Not CKD = 0
target_map = {"ckd": 1, "notckd": 0}
y = df["class"].map(target_map)

if y.isna().any():
    raise ValueError("Unexpected values found in target column 'class'.")

X = df.drop(columns=["class"])

assert X.shape[1] == 24

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Predictors:", X.shape[1])
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Target distribution:")
print(y.value_counts().rename(index={1: "ckd", 0: "notckd"}))


Predictors: 24
Numerical features: 14
Categorical features: 10
Target distribution:
class
ckd       250
notckd    150
Name: count, dtype: int64


In [6]:
# 4. Stratified 80:20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Train class distribution:")
print(y_train.value_counts())
print("Test class distribution:")
print(y_test.value_counts())


Training records: 320
Test records: 80
Train class distribution:
class
1    200
0    120
Name: count, dtype: int64
Test class distribution:
class
1    50
0    30
Name: count, dtype: int64


In [7]:
# 5. Leakage-safe preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Numerical: median imputation + StandardScaler")
print("Categorical: most-frequent imputation + OneHotEncoder")
print("Preprocessing will be fitted inside each CV training fold.")


Numerical: median imputation + StandardScaler
Categorical: most-frequent imputation + OneHotEncoder
Preprocessing will be fitted inside each CV training fold.


In [8]:
# 6. Define the 9 models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=7,
        weights="distance"
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=75,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=75,
        learning_rate=0.05,
        max_depth=3,
        min_samples_split=5,
        min_samples_leaf=2,
        subsample=0.9,
        random_state=RANDOM_STATE
    )
}

# Optional extension models
if importlib.util.find_spec("xgboost") is not None:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=2,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

if importlib.util.find_spec("lightgbm") is not None:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        num_leaves=15,
        min_child_samples=10,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        verbosity=-1,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

print("Models available:")
for i, name in enumerate(models, 1):
    print(f"{i}. {name}")
assert len(models) == 9, f"Expected 9 models, found {len(models)}"


Models available:
1. Logistic Regression
2. Decision Tree
3. KNN
4. SVM
5. Random Forest
6. AdaBoost
7. Gradient Boosting
8. XGBoost
9. LightGBM


In [9]:
# 7. Small, controlled hyperparameter search spaces
param_distributions = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 10.0],
        "model__solver": ["liblinear", "lbfgs"]
    },
    "Decision Tree": {
        "model__max_depth": [3, 5, 7],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },
    "SVM": {
        "model__C": [0.1, 1.0, 10.0],
        "model__gamma": ["scale", 0.01, 0.1]
    },
    "Random Forest": {
        "model__n_estimators": [50, 75, 100],
        "model__max_depth": [4, 6, 8],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2"]
    },
    "AdaBoost": {
        "model__n_estimators": [50, 75, 100],
        "model__learning_rate": [0.05, 0.1, 0.5, 1.0]
    },
    "Gradient Boosting": {
        "model__n_estimators": [50, 75, 100],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "XGBoost": {
        "model__n_estimators": [50, 75, 100],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3],
        "model__min_child_weight": [1, 2, 4],
        "model__subsample": [0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.8, 0.9, 1.0]
    },
    "LightGBM": {
        "model__n_estimators": [50, 75, 100],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
        "model__num_leaves": [7, 15, 31],
        "model__min_child_samples": [5, 10, 15]
    }
}

print("Each algorithm will test exactly 3 randomly sampled configurations.")
print(f"Maximum CV fits: {len(models)} × {N_ITER} × {CV_FOLDS} = {len(models)*N_ITER*CV_FOLDS}")


Each algorithm will test exactly 3 randomly sampled configurations.
Maximum CV fits: 9 × 3 × 5 = 135


In [10]:
# 8. Run tuning — 3 configurations per algorithm
cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

tuning_rows = []
best_estimators = {}

for model_name, model in models.items():
    print(f"\n{'='*65}")
    print(f"Tuning: {model_name}")
    print(f"{'='*65}")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions[model_name],
        n_iter=N_ITER,
        scoring="f1",
        cv=cv_strategy,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True,
        return_train_score=True
    )

    search.fit(X_train, y_train)

    best_estimators[model_name] = search.best_estimator_

    best_idx = search.best_index_
    mean_train_f1 = search.cv_results_["mean_train_score"][best_idx]
    mean_cv_f1 = search.best_score_
    gap = mean_train_f1 - mean_cv_f1

    tuning_rows.append({
        "Model": model_name,
        "Best_CV_F1": mean_cv_f1,
        "Mean_Train_F1": mean_train_f1,
        "Train_CV_Gap": gap,
        "Best_Params": str(search.best_params_)
    })

    print(f"Best CV F1: {mean_cv_f1:.4f}")
    print(f"Mean Train F1: {mean_train_f1:.4f}")
    print(f"Train-CV Gap: {gap:.4f}")
    print("Best parameters:", search.best_params_)

tuning_df = pd.DataFrame(tuning_rows).sort_values(
    "Best_CV_F1", ascending=False
).reset_index(drop=True)

print("\nTUNING COMPLETE")
display(tuning_df[["Model", "Best_CV_F1", "Mean_Train_F1", "Train_CV_Gap"]])



Tuning: Logistic Regression
Best CV F1: 0.9951
Mean Train F1: 1.0000
Train-CV Gap: 0.0049
Best parameters: {'model__solver': 'lbfgs', 'model__C': 10.0}

Tuning: Decision Tree
Best CV F1: 0.9752
Mean Train F1: 0.9975
Train-CV Gap: 0.0223
Best parameters: {'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_depth': 5}

Tuning: KNN
Best CV F1: 0.9793
Mean Train F1: 0.9873
Train-CV Gap: 0.0081
Best parameters: {'model__weights': 'uniform', 'model__p': 1, 'model__n_neighbors': 3}

Tuning: SVM
Best CV F1: 0.9949
Mean Train F1: 1.0000
Train-CV Gap: 0.0051
Best parameters: {'model__gamma': 0.1, 'model__C': 1.0}

Tuning: Random Forest
Best CV F1: 0.9951
Mean Train F1: 0.9981
Train-CV Gap: 0.0031
Best parameters: {'model__n_estimators': 75, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 6}

Tuning: AdaBoost
Best CV F1: 0.9925
Mean Train F1: 1.0000
Train-CV Gap: 0.0075
Best parameters: {'model__n_estimators': 50, 'model__learning_rate': 1.0}

,Model,Best_CV_F1,Mean_Train_F1,Train_CV_Gap
0,Logistic Regression,0.995122,1.000000,0.004878
1,Random Forest,0.995062,0.998131,0.003069
2,SVM,0.994937,1.000000,0.005063
3,AdaBoost,0.992468,1.000000,0.007532
4,XGBoost,0.990123,0.998758,0.008634
5,LightGBM,0.987404,1.000000,0.012596
6,Gradient Boosting,0.982654,0.995650,0.012996
7,KNN,0.979282,0.987334,0.008051
8,Decision Tree,0.975233,0.997500,0.022267


In [11]:
# 9. Generalization diagnostic
def generalization_note(row):
    gap = row["Train_CV_Gap"]
    train_f1 = row["Mean_Train_F1"]
    cv_f1 = row["Best_CV_F1"]

    if gap >= 0.05:
        return "Potential overfitting: large train-CV gap"
    elif train_f1 < 0.85 and cv_f1 < 0.85:
        return "Possible underfitting: both scores are low"
    else:
        return "No strong warning from train-CV gap"

tuning_df["Generalization_Diagnostic"] = tuning_df.apply(
    generalization_note, axis=1
)

display(tuning_df[
    ["Model", "Best_CV_F1", "Mean_Train_F1",
     "Train_CV_Gap", "Generalization_Diagnostic"]
])


,Model,Best_CV_F1,Mean_Train_F1,Train_CV_Gap,Generalization_Diagnostic
0,Logistic Regression,0.995122,1.000000,0.004878,No strong warning from train-CV gap
1,Random Forest,0.995062,0.998131,0.003069,No strong warning from train-CV gap
2,SVM,0.994937,1.000000,0.005063,No strong warning from train-CV gap
3,AdaBoost,0.992468,1.000000,0.007532,No strong warning from train-CV gap
4,XGBoost,0.990123,0.998758,0.008634,No strong warning from train-CV gap
5,LightGBM,0.987404,1.000000,0.012596,No strong warning from train-CV gap
6,Gradient Boosting,0.982654,0.995650,0.012996,No strong warning from train-CV gap
7,KNN,0.979282,0.987334,0.008051,No strong warning from train-CV gap
8,Decision Tree,0.975233,0.997500,0.022267,No strong warning from train-CV gap


In [12]:
# 10. Final evaluation on the untouched test set
evaluation_rows = []

for model_name, estimator in best_estimators.items():
    y_pred = estimator.predict(X_test)

    if hasattr(estimator, "predict_proba"):
        y_score = estimator.predict_proba(X_test)[:, 1]
    elif hasattr(estimator, "decision_function"):
        y_score = estimator.decision_function(X_test)
    else:
        y_score = y_pred

    evaluation_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_score)
    })

evaluation_df = pd.DataFrame(evaluation_rows).sort_values(
    "F1", ascending=False
).reset_index(drop=True)

print("Final test-set evaluation:")
display(evaluation_df)


Final test-set evaluation:


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Random Forest,1.0000,1.000000,1.00,1.000000,1.000000
1,XGBoost,1.0000,1.000000,1.00,1.000000,1.000000
2,LightGBM,1.0000,1.000000,1.00,1.000000,1.000000
3,Gradient Boosting,0.9875,0.980392,1.00,0.990099,1.000000
4,Decision Tree,0.9875,1.000000,0.98,0.989899,0.990000
5,AdaBoost,0.9875,1.000000,0.98,0.989899,1.000000
6,SVM,0.9875,1.000000,0.98,0.989899,1.000000
7,Logistic Regression,0.9750,1.000000,0.96,0.979592,0.999333
8,KNN,0.9625,1.000000,0.94,0.969072,0.990000


In [13]:
# 11. Detailed classification reports and confusion matrices
for model_name, estimator in best_estimators.items():
    y_pred = estimator.predict(X_test)

    print("\n" + "="*70)
    print(model_name)
    print("="*70)
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Not CKD", "CKD"],
        zero_division=0
    ))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))



Logistic Regression
              precision    recall  f1-score   support

     Not CKD       0.94      1.00      0.97        30
         CKD       1.00      0.96      0.98        50

    accuracy                           0.97        80
   macro avg       0.97      0.98      0.97        80
weighted avg       0.98      0.97      0.98        80

Confusion Matrix:
[[30  0]
 [ 2 48]]

Decision Tree
              precision    recall  f1-score   support

     Not CKD       0.97      1.00      0.98        30
         CKD       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80

Confusion Matrix:
[[30  0]
 [ 1 49]]

KNN
              precision    recall  f1-score   support

     Not CKD       0.91      1.00      0.95        30
         CKD       1.00      0.94      0.97        50

    accuracy                           0.96        80
   macro avg 

In [14]:
# 12. Combined tuning + final evaluation table
final_results = tuning_df.merge(
    evaluation_df,
    on="Model",
    how="left"
)

final_results = final_results.sort_values(
    "F1", ascending=False
).reset_index(drop=True)

display(final_results[
    ["Model", "Best_CV_F1", "Accuracy", "Precision",
     "Recall", "F1", "ROC_AUC", "Train_CV_Gap",
     "Generalization_Diagnostic"]
])


,Model,Best_CV_F1,Accuracy,Precision,Recall,F1,ROC_AUC,Train_CV_Gap,Generalization_Diagnostic
0,Random Forest,0.995062,1.0000,1.000000,1.00,1.000000,1.000000,0.003069,No strong warning from train-CV gap
1,LightGBM,0.987404,1.0000,1.000000,1.00,1.000000,1.000000,0.012596,No strong warning from train-CV gap
2,XGBoost,0.990123,1.0000,1.000000,1.00,1.000000,1.000000,0.008634,No strong warning from train-CV gap
3,Gradient Boosting,0.982654,0.9875,0.980392,1.00,0.990099,1.000000,0.012996,No strong warning from train-CV gap
4,AdaBoost,0.992468,0.9875,1.000000,0.98,0.989899,1.000000,0.007532,No strong warning from train-CV gap
5,Decision Tree,0.975233,0.9875,1.000000,0.98,0.989899,0.990000,0.022267,No strong warning from train-CV gap
6,SVM,0.994937,0.9875,1.000000,0.98,0.989899,1.000000,0.005063,No strong warning from train-CV gap
7,Logistic Regression,0.995122,0.9750,1.000000,0.96,0.979592,0.999333,0.004878,No strong warning from train-CV gap
8,KNN,0.979282,0.9625,1.000000,0.94,0.969072,0.990000,0.008051,No strong warning from train-CV gap


In [15]:
# 13. Save results
results_dir = "../../results"
if not os.path.exists(results_dir):
    results_dir = "../results"
if not os.path.exists(results_dir):
    results_dir = "results"

os.makedirs(results_dir, exist_ok=True)

tuning_output = os.path.join(results_dir, "CKD_hyperparameter_tuning_results.csv")
evaluation_output = os.path.join(results_dir, "CKD_final_test_evaluation_after_tuning.csv")
combined_output = os.path.join(results_dir, "CKD_tuning_and_evaluation_combined.csv")

tuning_df.to_csv(tuning_output, index=False)
evaluation_df.to_csv(evaluation_output, index=False)
final_results.to_csv(combined_output, index=False)

print("Saved:")
print(tuning_output)
print(evaluation_output)
print(combined_output)


Saved:
../results\CKD_hyperparameter_tuning_results.csv
../results\CKD_final_test_evaluation_after_tuning.csv
../results\CKD_tuning_and_evaluation_combined.csv


## Final Interpretation

- Hyperparameter tuning uses **3 sampled configurations per algorithm**.
- Validation uses **5-fold Stratified Cross-Validation** on the training partition.
- The test partition is kept untouched during tuning and is used only for final evaluation.
- Numerical preprocessing uses median imputation and StandardScaler.
- Categorical preprocessing uses most-frequent imputation and one-hot encoding.
- The train–CV F1 gap is used only as a **diagnostic warning**, not as proof of overfitting.
- Final model selection should consider generalization and multiple evaluation metrics, not accuracy alone.

> **Important:** High accuracy on this 400-record dataset should not be interpreted as clinical validation. This is an academic machine-learning study, not a medical diagnostic system.
